In [0]:
%run ../00-common/config

In [0]:
%run ./00_silver_helpers

In [0]:
from pyspark.sql import functions as F

products = spark.table(f"{catalog_name}.{bronze_schema}.products").drop("ingestion_timestamp", "source_file", "batch_id")
category_tr = spark.table(f"{catalog_name}.{silver_schema}.category_tr")

products_silver = (
    products.join(category_tr, on="product_category_name", how="left")
    # rregullo gabimin 'lenght' ne 'length'
    .withColumnRenamed("product_name_lenght", "product_name_length")
    .withColumnRenamed("product_description_lenght", "product_description_length")
    # produktet pa kategori i bejme uncategorized
    .withColumn("product_category_name_english",
        F.coalesce(F.col("product_category_name_english"), F.lit("uncategorized")))
)
write_to_silver(products_silver, "products", catalog_name, silver_schema)